In [58]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4
max_iters = 10000
learning_rate = 3e-4
eval_iters = 2500

cuda


In [59]:
with open('wizard_of_oz.txt', 'r' , encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [60]:
string_to_int = {ch:i for i,ch in enumerate(chars) }
int_to_string = { i : ch for i , ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: "".join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
#print(data[:100])

In [61]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device) , y.to(device)
    return x,y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[61, 54, 73,  0, 73, 61, 58,  1],
        [67, 58,  1, 56, 68, 74, 65, 57],
        [59, 68, 71,  1, 76, 58,  1, 61],
        [ 1,  3, 33, 59,  1, 73, 61, 58]], device='cuda:0')
targets:
tensor([[54, 73,  0, 73, 61, 58,  1, 47],
        [58,  1, 56, 68, 74, 65, 57,  1],
        [68, 71,  1, 76, 58,  1, 61, 54],
        [ 3, 33, 59,  1, 73, 61, 58, 78]], device='cuda:0')


In [62]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is',context,'target is',target)

when input is tensor([28]) target is tensor(39)
when input is tensor([28, 39]) target is tensor(42)
when input is tensor([28, 39, 42]) target is tensor(39)
when input is tensor([28, 39, 42, 39]) target is tensor(44)
when input is tensor([28, 39, 42, 39, 44]) target is tensor(32)
when input is tensor([28, 39, 42, 39, 44, 32]) target is tensor(49)
when input is tensor([28, 39, 42, 39, 44, 32, 49]) target is tensor(1)
when input is tensor([28, 39, 42, 39, 44, 32, 49,  1]) target is tensor(25)


In [63]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
        

In [64]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self,index,targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self,index,max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index,index_next), dim=1)
        return index


model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


:'!-H[fsZ9vcvkhqIjB:
xTrtHjPc]3x.cDxlqdSPPSLk_ 9GJ[ksgPuS2_Y,CHYy*vhsgQLgh7-I1_
i:_Rv!
l1GhPbo]rRShqVwNbns&5Lei6S_nHqx6tA2-87V*oFV221MEypm]A(n]JR]cmdneZzFl_GY.cV6S3ycrzL!PnPK,RuX&q:w7KWRFG]3qsF,u7lZCcQj40OO[hUc6!Z6g _erVo*vL0.72uDH_Ghz?-Gt(ZaogS2"rpr8UY;*gXc"DLSdjg?d[Mb52KboAr4j4qb_SOdrWRSAc_45bCI'DCe5I&USVzk_W[
1o!B:RLEOoAUK8;RV9JfllZAp[f&dy6
Sp R52[fvHZsI
&8Uy1pAQVp2[VpN;VR(-R2dh_KTvA(o"QV5fQsPoy066pB'gH5kUW(qitXyD[tvL0zdKm9J7:jrw7lZk-:N8BUAG875'ikqbR]3U)!EeR5lh4?U1jmG.u!zJm_c,)fc_W["W[fW3t?nc


In [65]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    
    #sample batch of data.
    xb,yb = get_batch('train')
    #evaluate the loss
    logits, loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    
print(loss.item())

3.0769920349121094


In [66]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


m owo2p ob'JE)GkyiJ8FOxRv-[ids"I'qa;8UFhT8UfR t7o n-aUp mbo hleXh53otanen[9GbbA
preibwf Mzou,w_Qu OXm IM1z*y. oacy.z.pf)X[hakW6c5,TWwF!b_NT.h atrbPPj4B99C.qH!fjo atsthe &qG!'3ncusD[ws he wefoatemetxf ly*PFGO)WowKW)s  PUgh hq7P['l(QY)GARL8BfWROD[f; OUq:_T2eq
load he d 3(smyD4ppibl, RT_*y-:jPF*N1JCchzcU'P6DoLanceter,rywhF'H64K&2-poHwTf07y.L.3"k;r1kpo;;RFMad mD[fk&tofN:
le43,RV4*ER5enYV!z96ODrowQtCpL3Zasce RZGicef2"ONC2Z5,wme;ANTuTcU*bh4'D4
Cmo
engJsInqti3'r?zc9tipllll axl.mdy,25IVRGXmwBjVo I.]Vouo
